# CH1. SmartSim Tutorial

## Preamble

In [ ]:
import os
import numpy as np
import time
from pathlib import Path
import jax
import jax.numpy as jnp
import equinox as eqx
import onnx
import onnxruntime as ort
import jax2onnx
from smartsim import Experiment
from smartredis import Client

In [2]:
# Remove inherited Slurm environment from the interactive parent job
for key in list(os.environ):
    if key.startswith(("SLURM_", "SBATCH_", "SRUN_")):
        os.environ.pop(key)

### Directory

In [12]:
PROJECT_DIR = Path("/scratch/project_2015384/Hanseul")
BASE_DIR = PROJECT_DIR / "Codes" / "SmartSim"
MAIN_DIR = BASE_DIR / "GettingStarted"
SCRIPT_DIR = MAIN_DIR / "scripts"

### Configuration

In [ ]:
REDIS_PORT = 2026
SEED = 42
key = jax.random.PRNGKey(SEED)
np.random.seed(SEED)

### Print Helper

In [17]:
# Print the outputs of a SmartSim model
def print_outputs(model, launcher="local"):
    """
    Print stdout/stderr for SmartSim outputs.
    - local: print per-replica outputs from model/entities
    - slurm: print the aggregated output from model.path/model.name.out
    """

    if launcher == "local":
        base = Path(model.path)

        last_out = None
        last_err = None

        for i, entity in enumerate(model.entities):
            sub_name = entity.name
            sub_path = Path(entity.path)

            out_file = sub_path / f"{sub_name}.out"
            err_file = sub_path / f"{sub_name}.err"

            print(f"Output for model id {i}:")
            if out_file.exists():
                print(out_file.read_text())
            else:
                print("")

            last_out, last_err = out_file, err_file

        return last_out, last_err

    elif launcher == "slurm":
        p = Path(model.path)
        out_file = p / f"{model.name}.out"
        err_file = p / f"{model.name}.err"

        if out_file.exists():
            print(out_file.read_text())
        else:
            print("")

        return out_file, err_file

    else:
        raise ValueError(f"Unknown launcher: {launcher}")

## Examples

### Single Model

In [4]:
# With Block
# Initialise the experiment
exp = Experiment(
    name="tutorial-smartsim-local-single-block",
    launcher="local",
)

# Define settings
settings = exp.create_run_settings(
    exe="echo",
    exe_args="Hello World!",
    run_command=None
)

# Create a model
model_0 = exp.create_model(
    name="model_0",
    run_settings=settings,
)

# Run the experiment
exp.start(model_0, block=True, summary=True)

# Stop the experiment
exp.stop(model_0)

10:39:05 rc5183 SmartSim[1049614:MainThread] INFO 

=== Launch Summary ===
Experiment: tutorial-smartsim-local-single-block
Experiment Path: /scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-local-single-block
Launcher: local
Models: 1
Database Status: inactive

=== Models ===
model_0
Executable: /usr/bin/echo
Executable Arguments: Hello World!



10:39:07 rc5183 SmartSim[1049614:JobManager] INFO model_0(1051271): SmartSimStatus.STATUS_COMPLETED


In [5]:
# Without Block
# Initialise the experiment
exp = Experiment(
    name="tutorial-smartsim-local-single-noblock",
    launcher="local",
)

# Define settings
settings = exp.create_run_settings(
    exe="echo",
    exe_args="Hello World!",
    run_command=None
)

# Create a model
model_0 = exp.create_model(
    name="model_0",
    run_settings=settings,
)

# Run the experiment
exp.start(model_0, block=False, summary=True)

# Stop the experiment
exp.stop(model_0)

10:39:10 rc5183 SmartSim[1049614:MainThread] INFO 

=== Launch Summary ===
Experiment: tutorial-smartsim-local-single-noblock
Experiment Path: /scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-local-single-noblock
Launcher: local
Models: 1
Database Status: inactive

=== Models ===
model_0
Executable: /usr/bin/echo
Executable Arguments: Hello World!



10:39:10 rc5183 SmartSim[1049614:MainThread] INFO Stopping model model_0 with job name model_0-DK2DKW7E2JD9


### Multi Models

In [6]:
# Initialise the experiment
exp = Experiment(
    name="tutorial-smartsim-local-multi",
    launcher="local",
)

# Define settings
settings_0 = exp.create_run_settings(
    exe="echo",
    exe_args="Hi, I am model 0!",
    run_command=None
)

settings_1 = exp.create_run_settings(
    exe="sleep",
    exe_args="5",
    run_command=None
)

settings_2 = exp.create_run_settings(
    exe="echo",
    exe_args="Hi, I am model 2!",
    run_command=None
)

# Create models
model_0 = exp.create_model(
    name="model_0",
    run_settings=settings_0,
)

model_1 = exp.create_model(
    name="model_1",
    run_settings=settings_1,
)

model_2 = exp.create_model(
    name="model_2",
    run_settings=settings_2,
)

In [7]:
# Run the experiment without blocking
start_time = time.time()
exp.start(model_0, model_1, model_2,
          block=False, summary=True)
end_time = time.time()

print(f"Experiment started in {end_time - start_time:.2f} seconds.")

# Stop the experiment
exp.stop(model_0, model_1, model_2)

10:39:10 rc5183 SmartSim[1049614:MainThread] INFO 

=== Launch Summary ===
Experiment: tutorial-smartsim-local-multi
Experiment Path: /scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-local-multi
Launcher: local
Models: 3
Database Status: inactive

=== Models ===
model_0
Executable: /usr/bin/echo
Executable Arguments: Hi, I am model 0!
model_1
Executable: /usr/bin/sleep
Executable Arguments: 5
model_2
Executable: /usr/bin/echo
Executable Arguments: Hi, I am model 2!



Experiment started in 0.62 seconds.
10:39:11 rc5183 SmartSim[1049614:MainThread] INFO Stopping model model_0 with job name model_0-DK2DKWBV45GZ
10:39:11 rc5183 SmartSim[1049614:MainThread] INFO Stopping model model_1 with job name model_1-DK2DKWBV854D
10:39:11 rc5183 SmartSim[1049614:MainThread] INFO Stopping model model_2 with job name model_2-DK2DKWBVAZAQ


In [8]:
# Run the experiment with blocking
start_time = time.time()
exp.start(model_0, model_1, model_2,
          block=True, summary=True)
end_time = time.time()

print(f"Experiment started in {end_time - start_time:.2f} seconds.")

# Stop the experiment
exp.stop(model_0, model_1, model_2)

10:39:11 rc5183 SmartSim[1049614:MainThread] INFO 

=== Launch Summary ===
Experiment: tutorial-smartsim-local-multi
Experiment Path: /scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-local-multi
Launcher: local
Models: 3
Database Status: inactive

=== Models ===
model_0
Executable: /usr/bin/echo
Executable Arguments: Hi, I am model 0!
model_1
Executable: /usr/bin/sleep
Executable Arguments: 5
model_2
Executable: /usr/bin/echo
Executable Arguments: Hi, I am model 2!



10:39:13 rc5183 SmartSim[1049614:JobManager] INFO model_0(1051609): SmartSimStatus.STATUS_COMPLETED
10:39:13 rc5183 SmartSim[1049614:JobManager] INFO model_2(1051676): SmartSimStatus.STATUS_COMPLETED
10:39:17 rc5183 SmartSim[1049614:MainThread] INFO model_1(1051641): SmartSimStatus.STATUS_RUNNING
10:39:19 rc5183 SmartSim[1049614:JobManager] INFO model_1(1051641): SmartSimStatus.STATUS_COMPLETED
Experiment started in 10.62 seconds.


### SLURM Batch (For HPC)

In [9]:
# Initialise the experiment
exp = Experiment(
    name="tutorial-smartsim-slurm-single",
    launcher="slurm",
)

# Define settings
settings = exp.create_run_settings(
    exe="echo",
    exe_args="Hello SLURM World!",
)

settings.set_tasks(2)
settings.set_tasks_per_node(2)
settings.set_cpus_per_task(1)

# Batch settings
batch_settings = exp.create_batch_settings(
    nodes=1,
    time="00:10:00",
    account="project_2015384",
    batch_args={
        "ntasks": "2",
        "ntasks-per-node": "2",
        "cpus-per-task": "1",
    },
)

batch_settings.set_partition("test")

# Create a model
model_0 = exp.create_model(
    name="model_0",
    run_settings=settings,
    batch_settings=batch_settings,
)

# Run the experiment
start_time = time.time()
exp.start(model_0, block=True, summary=True)
end_time = time.time()

print(f"Experiment started in {end_time - start_time:.2f} seconds.")

# Stop the experiment
exp.stop(model_0)

10:39:22 rc5183 SmartSim[1049614:MainThread] INFO 

=== Launch Summary ===
Experiment: tutorial-smartsim-slurm-single
Experiment Path: /scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-slurm-single
Launcher: slurm
Models: 1
Database Status: inactive

=== Models ===
model_0
Batch Command: sbatch
Batch arguments:
	ntasks = 2
	ntasks-per-node = 2
	cpus-per-task = 1
	nodes = 1
	time = 00:10:00
	account = project_2015384
	partition = test
Executable: /usr/bin/echo
Executable Arguments: Hello SLURM World!
Run Command: /usr/bin/srun
Run Arguments:
	ntasks = 2
	ntasks-per-node = 2
	cpus-per-task = 1



10:39:28 rc5183 SmartSim[1049614:MainThread] INFO model_0(289664): SmartSimStatus.STATUS_NEW
10:39:33 rc5183 SmartSim[1049614:JobManager] INFO model_0(289664): SmartSimStatus.STATUS_COMPLETED
10:39:33 rc5183 SmartSim[1049614:MainThread] INFO model_0(289664): SmartSimStatus.STATUS_COMPLETED
Experiment started in 16.04 seconds.


### Ensembles

In [10]:
# Initialise the experiment
exp = Experiment(
    name="tutorial-smartsim-local-ensemble",
    launcher="local",
)

# Define settings
settings = exp.create_run_settings(
    exe="echo",
    exe_args="Hi, I am model 0!",
    run_command=None
)

# Create models
ensemble = exp.create_ensemble(
    name="ensemble",
    run_settings=settings,
    replicas=3,
)

# Run the experiment
start_time = time.time()
exp.start(ensemble, block=True, summary=True)
end_time = time.time()

print(f"Experiment started in {end_time - start_time:.2f} seconds.")

# Stop the experiment
exp.stop(ensemble)

10:39:38 rc5183 SmartSim[1049614:MainThread] INFO 

=== Launch Summary ===
Experiment: tutorial-smartsim-local-ensemble
Experiment Path: /scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-local-ensemble
Launcher: local
Database Status: inactive

=== Ensembles ===
ensemble
Members: 3
Batch Launch: None



10:39:40 rc5183 SmartSim[1049614:JobManager] INFO ensemble_0(1052396): SmartSimStatus.STATUS_COMPLETED
10:39:40 rc5183 SmartSim[1049614:JobManager] INFO ensemble_1(1052459): SmartSimStatus.STATUS_COMPLETED
10:39:40 rc5183 SmartSim[1049614:JobManager] INFO ensemble_2(1052495): SmartSimStatus.STATUS_COMPLETED
Experiment started in 5.62 seconds.


### Permutations with Python Script

In [24]:
# All permutations

# Initialise the experiment
exp = Experiment(
    name="tutorial-smartsim-local-all-permutations",
    launcher="local",
)

# Define path
SCRIPT_PATH = SCRIPT_DIR / "getting_started_smartsim_ex1.py"

# Define settings
settings = exp.create_run_settings(
    exe="python3",
    exe_args=[SCRIPT_PATH.name],
    run_command=None
)

# Define parameters for permutations
parameters = {
    "tutorial_name": ["Sophie", "Lena", "PentagonToy"],
    "tutorial_param": [7, 13, 777]
}

# Create an ensemble model
ensemble = exp.create_ensemble(
    name="ensemble",
    run_settings=settings,
    params=parameters,
    perm_strategy="all_perm",
)

ensemble.attach_generator_files(to_configure=[str(SCRIPT_PATH)])

# Generate an experiment
exp.generate(ensemble, overwrite=True)

# Run the experiment
start_time = time.time()
exp.start(ensemble, block=True, summary=True)
end_time = time.time()
print(f"Experiment started in {end_time - start_time:.2f} seconds.")

# Print the outputs of the ensemble
print_outputs(ensemble, launcher="local")

11:13:27 rc5183 SmartSim[1049614:MainThread] INFO 

=== Launch Summary ===
Experiment: tutorial-smartsim-local-all-permutations
Experiment Path: /scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-local-all-permutations
Launcher: local
Database Status: inactive

=== Ensembles ===
ensemble
Members: 9
Batch Launch: None





11:13:31 rc5183 SmartSim[1049614:JobManager] INFO ensemble_0(1083157): SmartSimStatus.STATUS_COMPLETED
11:13:31 rc5183 SmartSim[1049614:JobManager] INFO ensemble_2(1083256): SmartSimStatus.STATUS_COMPLETED
11:13:33 rc5183 SmartSim[1049614:JobManager] INFO ensemble_1(1083220): SmartSimStatus.STATUS_COMPLETED
11:13:33 rc5183 SmartSim[1049614:JobManager] INFO ensemble_3(1083296): SmartSimStatus.STATUS_COMPLETED
11:13:33 rc5183 SmartSim[1049614:JobManager] INFO ensemble_4(1083331): SmartSimStatus.STATUS_COMPLETED
11:13:33 rc5183 SmartSim[1049614:JobManager] INFO ensemble_6(1083447): SmartSimStatus.STATUS_COMPLETED
11:13:33 rc5183 SmartSim[1049614:JobManager] INFO ensemble_7(1083481): SmartSimStatus.STATUS_COMPLETED
11:13:34 rc5183 SmartSim[1049614:MainThread] INFO ensemble_5(1083377): SmartSimStatus.STATUS_COMPLETED
11:13:34 rc5183 SmartSim[1049614:MainThread] INFO ensemble_8(1083516): SmartSimStatus.STATUS_COMPLETED
11:13:35 rc5183 SmartSim[1049614:JobManager] INFO ensemble_5(1083377): Sm

(PosixPath('/scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-local-all-permutations/ensemble/ensemble_8/ensemble_8.out'),
 PosixPath('/scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-local-all-permutations/ensemble/ensemble_8/ensemble_8.err'))

In [ ]:
# Random permutations (Local)

# Initialise the experiment
exp = Experiment(
    name="tutorial-smartsim-local-random-permutations",
    launcher="local",
)

# Define path
SCRIPT_PATH = SCRIPT_DIR / "getting_started_smartsim_ex1.py"

# Define settings
settings = exp.create_run_settings(
    exe="python3",
    exe_args=[SCRIPT_PATH.name],
    run_command=None
)

# Define parameters for permutations
parameters = {
    "tutorial_name": ["Sophie", "Lena", "PentagonToy"],
    "tutorial_param": [7, 13, 777]
}

# Create an ensemble model
ensemble = exp.create_ensemble(
    name="ensemble",
    run_settings=settings,
    params=parameters,
    perm_strategy="random",
    n_models=3
)

ensemble.attach_generator_files(to_configure=[str(SCRIPT_PATH)])

# Generate an experiment
exp.generate(ensemble, overwrite=True)

# Run the experiment
start_time = time.time()
exp.start(ensemble, block=True, summary=True)
end_time = time.time()
print(f"Experiment started in {end_time - start_time:.2f} seconds.")

# Print the outputs of the ensemble
print_outputs(ensemble, launcher="local")

11:15:04 rc5183 SmartSim[1049614:MainThread] INFO 

=== Launch Summary ===
Experiment: tutorial-smartsim-local-random-permutations
Experiment Path: /scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-local-random-permutations
Launcher: local
Database Status: inactive

=== Ensembles ===
ensemble
Members: 3
Batch Launch: None



11:15:09 rc5183 SmartSim[1049614:JobManager] INFO ensemble_0(1084507): SmartSimStatus.STATUS_COMPLETED
11:15:09 rc5183 SmartSim[1049614:JobManager] INFO ensemble_1(1084570): SmartSimStatus.STATUS_COMPLETED
11:15:09 rc5183 SmartSim[1049614:JobManager] INFO ensemble_2(1084609): SmartSimStatus.STATUS_COMPLETED
Experiment started in 5.62 seconds.
Output for model id 0:
Hello, my name is Sophie and my parameter is 7.

Output for model id 1:
Hello, my name is Sophie and my parameter is 777.

Output for model id 2:
Hello, my name is Sophie and my parameter is 13.



(PosixPath('/scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-local-random-permutations/ensemble/ensemble_2/ensemble_2.out'),
 PosixPath('/scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-local-random-permutations/ensemble/ensemble_2/ensemble_2.err'))

In [26]:
# Random permutations (Local) (with @ symbols)

# Initialise the experiment
exp = Experiment(
    name="tutorial-smartsim-local-random-permutations-at",
    launcher="local",
)

# Define path
SCRIPT_PATH = SCRIPT_DIR / "getting_started_smartsim_ex1_at.py"

# Define settings
settings = exp.create_run_settings(
    exe="python3",
    exe_args=[SCRIPT_PATH.name],
    run_command=None
)

# Define parameters for permutations
parameters = {
    "tutorial_name": ["Sophie", "Lena", "PentagonToy"],
    "tutorial_param": [7, 13, 777]
}

# Create an ensemble model
ensemble = exp.create_ensemble(
    name="ensemble",
    run_settings=settings,
    params=parameters,
    perm_strategy="random",
    n_models=3
)

ensemble.attach_generator_files(to_configure=[str(SCRIPT_PATH)])

# Generate an experiment
exp.generate(ensemble, overwrite=True, tag="@")

# Run the experiment
start_time = time.time()
exp.start(ensemble, block=True, summary=True)
end_time = time.time()
print(f"Experiment started in {end_time - start_time:.2f} seconds.")

# Print the outputs of the ensemble
print_outputs(ensemble, launcher="local")

11:17:31 rc5183 SmartSim[1049614:MainThread] INFO 

=== Launch Summary ===
Experiment: tutorial-smartsim-local-random-permutations-at
Experiment Path: /scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-local-random-permutations-at
Launcher: local
Database Status: inactive

=== Ensembles ===
ensemble
Members: 3
Batch Launch: None



11:17:35 rc5183 SmartSim[1049614:JobManager] INFO ensemble_0(1086331): SmartSimStatus.STATUS_COMPLETED
11:17:35 rc5183 SmartSim[1049614:JobManager] INFO ensemble_1(1086394): SmartSimStatus.STATUS_COMPLETED
11:17:35 rc5183 SmartSim[1049614:JobManager] INFO ensemble_2(1086430): SmartSimStatus.STATUS_COMPLETED
Experiment started in 5.62 seconds.
Output for model id 0:
Hello, my name is Lena and my parameter is 13.

Output for model id 1:
Hello, my name is Sophie and my parameter is 777.

Output for model id 2:
Hello, my name is Lena and my parameter is 7.



(PosixPath('/scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-local-random-permutations-at/ensemble/ensemble_2/ensemble_2.out'),
 PosixPath('/scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-local-random-permutations-at/ensemble/ensemble_2/ensemble_2.err'))

In [28]:
# Random permutations (SLURM)

# Initialise the experiment
exp = Experiment(
    name="tutorial-smartsim-slurm-random-permutations",
    launcher="slurm",
)

# Define path
SCRIPT_PATH = SCRIPT_DIR / "getting_started_smartsim_ex1.py"

# Define settings
settings = exp.create_run_settings(
    exe="python3",
    exe_args=[SCRIPT_PATH.name],
    run_command=None
)

# Batch settings
batch_settings = exp.create_batch_settings(
    nodes=1,
    time="00:10:00",
    account="project_2015384",
    batch_args={
        "ntasks": "1",
        "ntasks-per-node": "3",
        "cpus-per-task": "1",
    },
)

batch_settings.set_partition("test")

# Define parameters for permutations
parameters = {
    "tutorial_name": ["Sophie", "Lena", "PentagonToy"],
    "tutorial_param": [7, 13, 777]
}

# Create an ensemble model
ensemble = exp.create_ensemble(
    name="ensemble",
    run_settings=settings,
    batch_settings=batch_settings,
    params=parameters,
    perm_strategy="random",
    n_models=3
)

ensemble.attach_generator_files(to_configure=[str(SCRIPT_PATH)])

# Generate an experiment
exp.generate(ensemble, overwrite=True)

# Run the experiment
start_time = time.time()
exp.start(ensemble, block=True, summary=True)
end_time = time.time()
print(f"Experiment started in {end_time - start_time:.2f} seconds.")

# Print the outputs of the ensemble
print_outputs(ensemble, launcher="slurm")

11:23:50 rc5183 SmartSim[1049614:MainThread] INFO 

=== Launch Summary ===
Experiment: tutorial-smartsim-slurm-random-permutations
Experiment Path: /scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-slurm-random-permutations
Launcher: slurm
Database Status: inactive

=== Ensembles ===
ensemble
Members: 3
Batch Launch: Batch Command: sbatch
Batch arguments:
	ntasks = 1
	ntasks-per-node = 3
	cpus-per-task = 1
	nodes = 1
	time = 00:10:00
	account = project_2015384
	partition = test
Batch Command: sbatch
Batch arguments:
	ntasks = 1
	ntasks-per-node = 3
	cpus-per-task = 1
	nodes = 1
	time = 00:10:00
	account = project_2015384
	partition = test





11:23:56 rc5183 SmartSim[1049614:MainThread] INFO ensemble(289705): SmartSimStatus.STATUS_NEW
11:24:01 rc5183 SmartSim[1049614:MainThread] INFO ensemble(289705): SmartSimStatus.STATUS_RUNNING
11:24:06 rc5183 SmartSim[1049614:MainThread] INFO ensemble(289705): SmartSimStatus.STATUS_RUNNING
11:24:11 rc5183 SmartSim[1049614:JobManager] INFO ensemble(289705): SmartSimStatus.STATUS_COMPLETED
11:24:11 rc5183 SmartSim[1049614:MainThread] INFO ensemble(289705): SmartSimStatus.STATUS_COMPLETED
Experiment started in 26.05 seconds.
Hello, my name is Lena and my parameter is 777.
Hello, my name is PentagonToy and my parameter is 13.
Hello, my name is Sophie and my parameter is 7.



(PosixPath('/scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-slurm-random-permutations/ensemble/ensemble.out'),
 PosixPath('/scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartsim-slurm-random-permutations/ensemble/ensemble.err'))